# Convert Alice EEG Dataset to BIDS

This notebook converts the raw Alice EEG dataset in BrainVision format into BIDS EEG format.

Source dataset:
- `/Users/yanyuwoo/Data/r`

Output BIDS dataset:
- `/Users/yanyuwoo/Data/bids`

Run the cells from top to bottom.

(This script already executed, move to this project for reference)

## 1. Imports

If `mne` or `mne-bids` is missing, install them in the notebook kernel environment first.

In [ ]:
from pathlib import Path
import json

import mne
from mne_bids import BIDSPath, write_raw_bids

## 2. Configure input and output paths

In [ ]:
RAW_DATA_ROOT = Path('/Users/yanyuwoo/Data/r')
BIDS_ROOT = Path('/Users/yanyuwoo/Data/bids')

print('Raw data root:', RAW_DATA_ROOT)
print('BIDS output root:', BIDS_ROOT)
print('Raw data exists:', RAW_DATA_ROOT.exists())
print('BIDS output exists:', BIDS_ROOT.exists())

## 3. Check that the BrainVision files are present

In [ ]:
subjects = [f'{i:02d}' for i in range(1, 50)]
missing = []

for subject in subjects:
    for ext in ('.vhdr', '.eeg', '.vmrk'):
        path = RAW_DATA_ROOT / f'S{subject}{ext}'
        if not path.exists():
            missing.append(str(path))

print('Subjects expected:', len(subjects))
print('Missing files:', len(missing))
missing[:10]

## 4. Inspect one raw file before conversion

In [ ]:
sample_vhdr = RAW_DATA_ROOT / 'S01.vhdr'
raw = mne.io.read_raw_brainvision(sample_vhdr, preload=False)
raw

## 5. Convert all subjects to BIDS

This will create a BIDS EEG dataset under `/Users/yanyuwoo/Data/bids`.

If you want to rerun the conversion from scratch, keep `overwrite=True`.

In [ ]:
BIDS_ROOT.mkdir(parents=True, exist_ok=True)

for subject in subjects:
    vhdr_file = RAW_DATA_ROOT / f'S{subject}.vhdr'
    print(f'Converting subject {subject}: {vhdr_file.name}')

    raw = mne.io.read_raw_brainvision(vhdr_file, preload=False)

    bids_path = BIDSPath(
        subject=subject,
        task='alice',
        datatype='eeg',
        root=BIDS_ROOT,
    )

    write_raw_bids(
        raw,
        bids_path,
        overwrite=True,
        format='BrainVision',
        allow_preload=False,
    )

print('Done.')

## 6. Inspect the generated BIDS structure

In [ ]:
top_level = sorted(p.name for p in BIDS_ROOT.iterdir())
top_level[:20]

In [ ]:
sub01_eeg = BIDS_ROOT / 'sub-01' / 'eeg'
sorted(p.name for p in sub01_eeg.iterdir())

## 7. Optional: write a simple dataset description patch

Usually `write_raw_bids()` creates `dataset_description.json`. This cell lets you inspect it.

In [ ]:
dataset_description = BIDS_ROOT / 'dataset_description.json'
print(dataset_description)
print(dataset_description.exists())

if dataset_description.exists():
    with open(dataset_description, 'r') as f:
        print(json.dumps(json.load(f), indent=2))

## Notes

- This notebook only converts the raw EEG recordings to BIDS.
- It does not yet generate TRF predictors.
- After conversion, later analysis code should point to `/Users/yanyuwoo/Data/bids`, not to the old path from the error message.